# ⚛️ Day 2 – Finance Basics: Data Collection & EDA
## Quantum Portfolio Optimization: Bank Nifty Stocks

### Learning Objectives for Day 2:
1. **Download historical price data** for 5–8 Bank Nifty stocks (2 years)
2. **Compute daily log returns** and understand time-series stationarity
3. **Calculate covariance matrix (Σ)** — how stocks move together
4. **Compute correlation matrix** — normalized co-movement (−1 to +1)
5. **Visualize correlation heatmap** — identify diversification opportunities

### Why This Matters for QAOA:
The QUBO (Quadratic Unconstrained Binary Optimization) problem solved by QAOA requires:
- **μ** (mean returns) — the "reward" for selecting each asset
- **Σ** (covariance matrix) — the "risk penalty" term

This notebook prepares both. Once these are saved, we'll move to **Day 3: QUBO Formulation**.

---

## Section 1: Import Required Libraries

In [10]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings

# Suppress yfinance warnings
warnings.filterwarnings('ignore')

# Set style for better-looking plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✔ All libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

✔ All libraries imported successfully!
Pandas version: 2.2.2
NumPy version: 2.0.2


## Section 2: Define Bank Nifty Stock Universe

In [11]:
# Bank Nifty constituents: 6 major Indian banking stocks
# (NSE tickers with .NS suffix for Yahoo Finance)
tickers = [
    "HDFCBANK.NS",      # HDFC Bank
    "ICICIBANK.NS",     # ICICI Bank
    "SBIN.NS",          # State Bank of India
    "AXISBANK.NS",      # Axis Bank
    "KOTAKBANK.NS",     # Kotak Mahindra Bank
    "INDUSINDBK.NS"     # IndusInd Bank
]

# Date range: 2 years of historical data (trading days)
end_date = datetime.now()
start_date = end_date - timedelta(days=2*365)

print("=" * 70)
print("BANK NIFTY STOCK UNIVERSE FOR DAY 2")
print("=" * 70)
print(f"✔ Selected Stocks: {', '.join(tickers)}")
print(f"✔ Data Period: {start_date.date()} → {end_date.date()}")
print(f"✔ Total trading days: ~{2*252}")
print("=" * 70)

BANK NIFTY STOCK UNIVERSE FOR DAY 2
✔ Selected Stocks: HDFCBANK.NS, ICICIBANK.NS, SBIN.NS, AXISBANK.NS, KOTAKBANK.NS, INDUSINDBK.NS
✔ Data Period: 2024-04-06 → 2026-04-06
✔ Total trading days: ~504


### Section 2A: Fetch Stock Price Data

We'll download **Adjusted Close (Adj Close)** prices from Yahoo Finance:
- **Adj Close**: Accounts for stock splits & dividend distributions
- **Daily frequency**: One observation per trading day
- **2 years**: Sufficient for computing stable covariance estimates

In [13]:
# Download historical price data for all tickers
import os
import glob

print("\n" + "=" * 70)
print("STEP 1: FETCH HISTORICAL PRICE DATA")
print("=" * 70)

prices_dict = {}

# Try to download from yfinance first
for ticker in tickers:
    print(f"\n📥 Downloading {ticker}...", end=" ")
    try:
        data = yf.download(
            ticker, 
            start=start_date.strftime("%Y-%m-%d"),
            end=end_date.strftime("%Y-%m-%d"),
            progress=False
        )
        
        # Handle yfinance output (may be empty or have different structure)
        if data.empty:
            print(f"✗ No data returned")
            continue
        
        # Flatten multi-level columns if needed
        if isinstance(data.columns, pd.MultiIndex):
            data.columns = data.columns.get_level_values(0)
        
        # Extract Adjusted Close price
        if 'Adj Close' in data.columns:
            adj_close = data['Adj Close']
        elif 'Close' in data.columns:
            adj_close = data['Close']
        else:
            print(f"✗ No price column found. Available: {list(data.columns)[:3]}")
            continue
        
        if adj_close.empty:
            print(f"✗ Price series is empty")
            continue
            
        prices_dict[ticker.replace('.NS', '')] = adj_close
        print(f"✔ ({len(adj_close)} trading days)")
    except Exception as e:
        print(f"✗ Error, trying fallback...")

# If yfinance downloads didn't work, load from cached CSV files
if not prices_dict:
    print("\n⚠️  yfinance downloads failed. Loading from data/raw/ CSV files...")
    
    # Direct path to raw data
    raw_dir = '/Users/johngeorgealexander/qc/quantum-portfolio-opt/data/raw'
    csv_files = sorted(glob.glob(os.path.join(raw_dir, '*.csv')))
    
    print(f"   Looking in: {raw_dir}")
    print(f"   Found {len(csv_files)} CSV files")
    
    for csv_file in csv_files:
        ticker_name = os.path.basename(csv_file).replace('.csv', '')
        try:
            df = pd.read_csv(csv_file, index_col=0, parse_dates=True)
            
            # Flatten multi-level columns if present
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            
            # Get the price column
            if 'Adj Close' in df.columns:
                prices_dict[ticker_name] = df['Adj Close']
            elif 'Close' in df.columns:
                prices_dict[ticker_name] = df['Close']
            else:
                # Take the second column (usually Close or Adj Close)
                prices_dict[ticker_name] = df.iloc[:, 4]  # Adj Close is typically 5th column
                
            print(f"   ✔ Loaded {ticker_name:12} ({len(prices_dict[ticker_name]):4} days)")
        except Exception as e:
            print(f"   ✗ {ticker_name}: {str(e)[:35]}")

# Combine into a single DataFrame with aligned dates
if prices_dict:
    prices_df = pd.DataFrame(prices_dict)
    prices_df.sort_index(inplace=True)
    
    # Handle missing data
    prices_df.ffill(inplace=True)  # Forward fill
    prices_df.dropna(inplace=True)  # Drop remaining NaNs
    
    print("\n" + "─" * 70)
    print("PRICE DATA SUMMARY")
    print("─" * 70)
    print(f"✔ Shape: {prices_df.shape[0]} trading days × {prices_df.shape[1]} stocks")
    if prices_df.shape[0] > 0:
        print(f"✔ Date range: {prices_df.index[0].date()} → {prices_df.index[-1].date()}")
    print(f"✔ Stocks: {list(prices_df.columns)}")
    
    # Display last 5 rows
    print("\nLast 5 trading days (Adjusted Close Prices):")
    print(prices_df.tail())
else:
    print("\n✗ CRITICAL: Unable to load any price data!")
    prices_df = pd.DataFrame()
    
print("=" * 70)


STEP 1: FETCH HISTORICAL PRICE DATA

📥 Downloading HDFCBANK.NS... ✔ (492 trading days)

📥 Downloading ICICIBANK.NS... ✔ (492 trading days)

📥 Downloading SBIN.NS... ✔ (492 trading days)

📥 Downloading AXISBANK.NS... ✔ (492 trading days)

📥 Downloading KOTAKBANK.NS... ✔ (492 trading days)

📥 Downloading INDUSINDBK.NS... ✔ (492 trading days)

──────────────────────────────────────────────────────────────────────
PRICE DATA SUMMARY
──────────────────────────────────────────────────────────────────────
✔ Shape: 492 trading days × 6 stocks
✔ Date range: 2024-04-08 → 2026-04-02
✔ Stocks: ['HDFCBANK', 'ICICIBANK', 'SBIN', 'AXISBANK', 'KOTAKBANK', 'INDUSINDBK']

Last 5 trading days (Adjusted Close Prices):
              HDFCBANK    ICICIBANK         SBIN     AXISBANK   KOTAKBANK  \
Date                                                                        
2026-03-25  782.299988  1259.699951  1060.599976  1222.099976  371.100006   
2026-03-27  756.200012  1233.800049  1019.500000  1205.19995

### Section 2B: Visualize Price Time Series

A quick plot to see how each stock has evolved over the past 2 years:

In [9]:
if not prices_df.empty and prices_df.shape[0] > 0:
    # Normalize prices (base 100) for easy comparison across different price scales
    prices_normalized = prices_df / prices_df.iloc[0] * 100

    fig, ax = plt.subplots(figsize=(14, 7))
    for col in prices_normalized.columns:
        ax.plot(prices_normalized.index, prices_normalized[col], label=col, linewidth=2, alpha=0.8)

    ax.set_title("Normalized Price Evolution (Base 100) – Bank Nifty Stocks (2 Years)", 
                 fontsize=14, fontweight="bold", pad=15)
    ax.set_xlabel("Date", fontsize=12)
    ax.set_ylabel("Normalized Price Index", fontsize=12)
    ax.legend(loc='best', fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    print("✔ Price chart displayed (normalized to ease cross-stock comparison)")
else:
    print("⚠️  Cannot plot prices: DataFrame is empty. Check data loading in previous cell.")

⚠️  Cannot plot prices: DataFrame is empty. Check data loading in previous cell.


## Section 3: Compute Daily Log Returns

**Why log returns?**
- **Time-additive**: Weekly return = sum of daily log returns (clean algebra)
- **Symmetry**: +100% gain ≡ −50% loss in magnitude
- **Normality**: Closer to normal distribution than simple percentage returns
- **QAOA-ready**: Most portfolio optimization models assume normally-distributed returns

**Formula**: $r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)$

In [ ]:
print("\n" + "=" * 70)
print("STEP 2: COMPUTE DAILY LOG RETURNS")
print("=" * 70)

if prices_df.empty or prices_df.shape[0] == 0:
    print("⚠️  Cannot compute returns: prices_df is empty!")
    returns_df = pd.DataFrame()
else:
    # Log returns: ln(P_t / P_{t-1})
    returns_df = np.log(prices_df / prices_df.shift(1)).dropna()

    print(f"\n✔ Computed daily log returns")
    print(f"   Shape: {returns_df.shape[0]} trading days × {returns_df.shape[1]} stocks")
    if returns_df.shape[0] > 0:
        print(f"   Date range: {returns_df.index[0].date()} → {returns_df.index[-1].date()}")

    # Display statistics
    print("\n" + "─" * 70)
    print("RETURN STATISTICS (Daily)")
    print("─" * 70)
    stats = pd.DataFrame({
        'Mean':     returns_df.mean(),
        'Std Dev':  returns_df.std(),
        'Min':      returns_df.min(),
        'Max':      returns_df.max(),
        'Skewness': returns_df.skew(),
    })
    print(stats.round(4))
    print("=" * 70)

    # Display last 5 rows of returns
    print("\nLast 5 trading days (Daily Log Returns):")
    print(returns_df.tail().round(6))

### Section 3A: Visualize Return Distributions

In [ ]:
if not returns_df.empty and returns_df.shape[0] > 0:
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()

    for idx, stock in enumerate(returns_df.columns):
        axes[idx].hist(returns_df[stock], bins=50, alpha=0.7, color='steelblue', edgecolor='black')
        axes[idx].axvline(returns_df[stock].mean(), color='red', linestyle='--', linewidth=2, label='Mean')
        axes[idx].set_title(f"{stock} – Daily Log Return Distribution", fontweight='bold')
        axes[idx].set_xlabel("Daily Log Return")
        axes[idx].set_ylabel("Frequency")
        axes[idx].legend()
        axes[idx].grid(True, alpha=0.3)

    plt.suptitle("Return Distribution Analysis – Bank Nifty Stocks", 
                 fontsize=14, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.show()

    print("✔ Return distributions visualized (aim for normality near zero)")
else:
    print("⚠️  Cannot plot return distributions: returns_df is empty!")

## Section 4: Compute Mean Returns and Covariance Matrix

### Mean Returns (μ) – The "Reward" Term

$\mu_i = \text{mean daily return}_i \times 252$

**Why multiply by 252?** There are ~252 trading days per year. To annualize daily returns, we scale by this factor.

### Covariance Matrix (Σ) – The "Risk" Term

$\Sigma_{ij} = \text{cov}(r_i, r_j) \times 252$

**Key properties:**
- **Diagonal** ($\Sigma_{ii}$): Variance of each stock (its own risk)
- **Off-diagonal** ($\Sigma_{ij}$): How stocks $i$ and $j$ move together
  - **Positive**: They tend to rise/fall together (bad for diversification)
  - **Near zero**: Independent movements (good for diversification)
  - **Negative**: They tend to move opposite (excellent for hedging)

In [ ]:
print("\n" + "=" * 70)
print("STEP 3: COMPUTE ANNUALISED MEAN RETURNS (μ) AND COVARIANCE (Σ)")
print("=" * 70)

if returns_df.empty or returns_df.shape[0] == 0:
    print("⚠️  Cannot compute statistics: returns_df is empty!")
    mu = pd.Series()
    sigma = pd.DataFrame()
else:
    # Annual scaling factor (trading days)
    annual_factor = 252

    # Mean returns (annualised)
    mu = returns_df.mean() * annual_factor

    print(f"\n✔ Annualised Mean Returns (μ)")
    print("─" * 70)
    print(mu.round(4))
    print("\nInterpretation: Expected annual return for each stock")

    # Covariance matrix (annualised)
    sigma = returns_df.cov() * annual_factor

    print(f"\n✔ Annualised Covariance Matrix (Σ)")
    print("─" * 70)
    print(sigma.round(4))
    print("\nInterpretation: How much variance (risk) in returns, both individual and co-movement")

    print("=" * 70)

## Section 5: Compute Correlation Matrix

**Correlation** is a **normalised covariance** — always in range $[-1, +1]$:

$$\rho_{ij} = \frac{\Sigma_{ij}}{\sigma_i \times \sigma_j}$$

**Interpretation:**
- **ρ = +1**: Perfect positive correlation (move together exactly)
- **ρ ≈ 0**: No linear relationship (good diversification candidate)
- **ρ = −1**: Perfect negative correlation (move opposite exactly)

In [ ]:
print("\n" + "=" * 70)
print("STEP 4: COMPUTE CORRELATION MATRIX")
print("=" * 70)

if returns_df.empty or returns_df.shape[0] == 0:
    print("⚠️  Cannot compute correlation: returns_df is empty!")
    corr = pd.DataFrame()
else:
    # Correlation matrix (daily returns)
    corr = returns_df.corr()

    print(f"\n✔ Correlation Matrix (Pearson correlation of daily log returns)")
    print("─" * 70)
    print(corr.round(4))

    print("\n✔ Key Observations:")
    print("─" * 70)

    # Find highest and lowest correlations (excluding diagonal)
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    corr_flat = corr.values[mask]

    print(f"   Highest correlation:  {corr_flat.max():.4f}")
    print(f"   Lowest correlation:   {corr_flat.min():.4f}")
    print(f"   Mean correlation:     {corr_flat.mean():.4f}")

    print("\n💡 Portfolio Insight:")
    if corr_flat.mean() > 0.6:
        print("   High average correlation → stocks move together")
        print("   → Limited diversification benefit from this subset")
    elif corr_flat.mean() > 0.4:
        print("   Moderate average correlation → mixed movement patterns")
        print("   → Some diversification benefit available")
    else:
        print("   Low average correlation → stocks move independently")
        print("   → Good diversification benefit from this subset")

    print("=" * 70)

## Section 6: Visualize Correlation Heatmap

This is the **main deliverable** for Day 2! The heatmap makes it easy to spot:
- **Clusters**: Groups of stocks that move together
- **Diversification candidates**: Pairs with low/negative correlation
- **Redundancy**: Highly correlated pairs (adding both = wasted diversification)

In [ ]:
print("\n" + "=" * 70)
print("STEP 5: CREATE CORRELATION HEATMAP")
print("=" * 70)

if corr.empty or corr.shape[0] == 0:
    print("⚠️  Cannot create heatmap: correlation matrix is empty!")
else:
    # Create the heatmap
    fig, ax = plt.subplots(figsize=(10, 8))

    # Use diverging color palette: red = high positive, blue = negative/low
    cmap = sns.diverging_palette(220, 10, as_cmap=True)

    sns.heatmap(
        corr,
        annot=True,              # Show correlation values
        fmt='.3f',               # 3 decimal places
        cmap=cmap,
        vmin=-1, vmax=1,         # Fix scale to [−1, +1]
        center=0,                # White = 0 (no correlation)
        square=True,             # Square cells
        linewidths=1,
        linecolor='white',
        ax=ax,
        cbar_kws={'shrink': 0.8, 'label': 'Pearson Correlation'},
        annot_kws={'size': 10, 'weight': 'bold'}
    )

    ax.set_title(
        'Day 2 – Correlation Heatmap\nBank Nifty Stocks (Daily Log Returns, 2 Years)',
        fontsize=14,
        fontweight='bold',
        pad=15
    )
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=11)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=11)

    plt.tight_layout()
    plt.show()

    print("\n✔ Correlation heatmap created and displayed")
    print("   Color scheme: Red = high positive | White = zero | Blue = negative")
    print("=" * 70)

## Section 7: Save Processed Data for Downstream Use

The computed **μ** (mean returns) and **Σ** (covariance) matrices will be fed directly into the QUBO builder on Day 3.

We save them in multiple formats:
- **CSV**: Human-readable for inspection
- **NumPy .npy**: Fast binary format for downstream scripts

In [ ]:
import os

print("\n" + "=" * 70)
print("STEP 6: SAVE PROCESSED DATA TO data/cached/")
print("=" * 70)

if prices_df.empty or returns_df.empty or mu.empty or corr.empty:
    print("⚠️  WARNING: One or more required data structures are empty!")
    print("   Cannot save artefacts. Check previous cells for errors.")
else:
    # Create the cached directory if it doesn't exist
    cached_dir = os.path.join(os.path.dirname(os.getcwd()), 'data', 'cached')
    os.makedirs(cached_dir, exist_ok=True)

    # Save mean returns
    mu.to_csv(os.path.join(cached_dir, 'expected_returns.csv'))
    np.save(os.path.join(cached_dir, 'mu.npy'), mu.values)

    # Save covariance matrix
    sigma.to_csv(os.path.join(cached_dir, 'covariance_matrix.csv'))
    np.save(os.path.join(cached_dir, 'sigma.npy'), sigma.values)

    # Save correlation matrix
    corr.to_csv(os.path.join(cached_dir, 'correlation_matrix.csv'))

    # Save full daily returns
    returns_df.to_csv(os.path.join(cached_dir, 'returns.csv'))

    print(f"\n✔ All artefacts saved to: {cached_dir}/")
    print("   Files created:")
    print("   • expected_returns.csv    – Annualised mean returns (μ)")
    print("   • mu.npy                  – NumPy array of μ (for QUBO builder)")
    print("   • covariance_matrix.csv   – Annualised covariance (Σ)")
    print("   • sigma.npy               – NumPy array of Σ (for QUBO builder)")
    print("   • correlation_matrix.csv  – Correlation matrix (ρ)")
    print("   • returns.csv             – Daily log returns (full history)")

    print("\n💡 Next Step (Day 3): QUBO Formulation")
    print("   Use μ and Σ to formulate the binary portfolio optimization problem")
    print("   Build the Q matrix and solve via brute force (ground truth)")
    print("=" * 70)


STEP 6: SAVE PROCESSED DATA TO data/cached/


NameError: name 'mu' is not defined